In [1]:
import pandas as pd
import numpy as np

# Load nav_history
df_nav = pd.read_csv("../data/raw/02_nav_history.csv")

print("BEFORE CLEANING:")
print(f"Shape: {df_nav.shape}")
print(f"Dtypes:\n{df_nav.dtypes}")
print(f"Nulls:\n{df_nav.isnull().sum()}")
print(f"\nSample:\n{df_nav.head()}")

BEFORE CLEANING:
Shape: (46000, 3)
Dtypes:
amfi_code      int64
date          object
nav          float64
dtype: object
Nulls:
amfi_code    0
date         0
nav          0
dtype: int64

Sample:
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [1]:
import pandas as pd
import numpy as np

# Load nav_history
df_nav = pd.read_csv("../data/raw/02_nav_history.csv")

# Step 1: Convert date to datetime
df_nav['date'] = pd.to_datetime(df_nav['date'])

# Step 2: Sort by amfi_code and date
df_nav = df_nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# Step 3: Remove duplicates
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])

# Step 4: Forward fill missing NAV (holidays/weekends) - fixed for newer pandas
df_nav = df_nav.set_index('date').groupby('amfi_code')[['nav']].resample('D').ffill().reset_index()

# Step 5: Validate NAV > 0
df_nav = df_nav[df_nav['nav'] > 0]

print("AFTER CLEANING:")
print(f"Shape: {df_nav.shape}")
print(f"Dtypes:\n{df_nav.dtypes}")
print(f"Nulls:\n{df_nav.isnull().sum()}")
print(f"\nSample:\n{df_nav.head()}")

# Save to processed folder
df_nav.to_csv("../data/processed/clean_nav.csv", index=False)
print("\n✅ Saved → data/processed/clean_nav.csv")

AFTER CLEANING:
Shape: (64320, 3)
Dtypes:
amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object
Nulls:
amfi_code    0
date         0
nav          0
dtype: int64

Sample:
   amfi_code       date       nav
0     100016 2022-01-03  520.4608
1     100016 2022-01-04  515.0971
2     100016 2022-01-05  521.7239
3     100016 2022-01-06  515.7880
4     100016 2022-01-07  515.1639

✅ Saved → data/processed/clean_nav.csv


In [2]:
# Load investor transactions
df_tx = pd.read_csv("../data/raw/08_investor_transactions.csv")

print("BEFORE CLEANING:")
print(f"Shape: {df_tx.shape}")
print(f"Dtypes:\n{df_tx.dtypes}")
print(f"Nulls:\n{df_tx.isnull().sum()}")
print(f"\nTransaction Types:\n{df_tx['transaction_type'].value_counts()}")
print(f"\nKYC Status:\n{df_tx['kyc_status'].value_counts()}")
print(f"\nSample:\n{df_tx.head(3)}")

BEFORE CLEANING:
Shape: (32778, 13)
Dtypes:
investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object
Nulls:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

KYC Status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

Sample:
  

In [4]:
# Step 1: Convert date to datetime
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# Step 2: Fix transaction_type properly
df_tx['transaction_type'] = df_tx['transaction_type'].str.strip()
df_tx['transaction_type'] = df_tx['transaction_type'].replace({
    'sip': 'SIP', 'Sip': 'SIP',
    'lumpsum': 'Lumpsum', 'LUMPSUM': 'Lumpsum',
    'redemption': 'Redemption', 'REDEMPTION': 'Redemption'
})

# Step 3: Validate amount > 0
df_tx = df_tx[df_tx['amount_inr'] > 0]

# Step 4: Standardise kyc_status
df_tx['kyc_status'] = df_tx['kyc_status'].str.strip().str.capitalize()

print("AFTER CLEANING:")
print(f"Shape: {df_tx.shape}")
print(f"Transaction Types:\n{df_tx['transaction_type'].value_counts()}")
print(f"KYC Status:\n{df_tx['kyc_status'].value_counts()}")

# Save
df_tx.to_csv("../data/processed/clean_transactions.csv", index=False)
print("\n✅ Saved → data/processed/clean_transactions.csv")

AFTER CLEANING:
Shape: (32778, 13)
Transaction Types:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64
KYC Status:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

✅ Saved → data/processed/clean_transactions.csv


In [5]:
# Load scheme performance
df_perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

print("BEFORE CLEANING:")
print(f"Shape: {df_perf.shape}")
print(f"Dtypes:\n{df_perf.dtypes}")
print(f"Nulls:\n{df_perf.isnull().sum()}")
print(f"\nSample:\n{df_perf.head(3)}")

BEFORE CLEANING:
Shape: (40, 19)
Dtypes:
amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object
Nulls:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_cror

In [6]:
# Step 1: Validate expense ratio range (0.1% - 2.5%)
invalid_expense = df_perf[
    (df_perf['expense_ratio_pct'] < 0.1) | 
    (df_perf['expense_ratio_pct'] > 2.5)
]
print(f"Funds with invalid expense ratio: {len(invalid_expense)}")

# Step 2: Flag negative Sharpe ratios
negative_sharpe = df_perf[df_perf['sharpe_ratio'] < 0]
print(f"Funds with negative Sharpe ratio: {len(negative_sharpe)}")

# Step 3: Validate max_drawdown is negative
invalid_drawdown = df_perf[df_perf['max_drawdown_pct'] > 0]
print(f"Funds with invalid max drawdown: {len(invalid_drawdown)}")

# Step 4: Validate return values are numeric (already confirmed)
print(f"\nExpense ratio range: {df_perf['expense_ratio_pct'].min()}% to {df_perf['expense_ratio_pct'].max()}%")
print(f"Sharpe ratio range: {df_perf['sharpe_ratio'].min()} to {df_perf['sharpe_ratio'].max()}")

# Save to processed folder
df_perf.to_csv("../data/processed/clean_performance.csv", index=False)
print("\n✅ Saved → data/processed/clean_performance.csv")

Funds with invalid expense ratio: 0
Funds with negative Sharpe ratio: 0
Funds with invalid max drawdown: 0

Expense ratio range: 0.55% to 1.64%
Sharpe ratio range: 0.8 to 7.68

✅ Saved → data/processed/clean_performance.csv


In [3]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

# Create database connection
engine = create_engine('sqlite:///bluestock_mf.db')
conn = sqlite3.connect('bluestock_mf.db')
cursor = conn.cursor()

# Read and execute schema.sql
with open('../sql/schema.sql', 'r') as f:
    schema = f.read()

cursor.executescript(schema)
conn.commit()
print("✅ Database created with all tables!")

# Load dim_fund
df_fund = pd.read_csv("../data/raw/01_fund_master.csv")
df_fund.to_sql('dim_fund', engine, if_exists='replace', index=False)
print(f"✅ dim_fund loaded: {len(df_fund)} rows")

# Load fact_nav
df_nav = pd.read_csv("../data/processed/clean_nav.csv")
df_nav.to_sql('fact_nav', engine, if_exists='replace', index=False)
print(f"✅ fact_nav loaded: {len(df_nav)} rows")

# Load fact_transactions
df_tx = pd.read_csv("../data/processed/clean_transactions.csv")
df_tx.to_sql('fact_transactions', engine, if_exists='replace', index=False)
print(f"✅ fact_transactions loaded: {len(df_tx)} rows")

# Load fact_performance
df_perf = pd.read_csv("../data/processed/clean_performance.csv")
df_perf.to_sql('fact_performance', engine, if_exists='replace', index=False)
print(f"✅ fact_performance loaded: {len(df_perf)} rows")

# Load fact_aum
df_aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
df_aum.to_sql('fact_aum', engine, if_exists='replace', index=False)
print(f"✅ fact_aum loaded: {len(df_aum)} rows")

print("\n🎉 All data loaded into bluestock_mf.db successfully!")
conn.close()

✅ Database created with all tables!
✅ dim_fund loaded: 40 rows
✅ fact_nav loaded: 64320 rows
✅ fact_transactions loaded: 32778 rows
✅ fact_performance loaded: 40 rows
✅ fact_aum loaded: 90 rows

🎉 All data loaded into bluestock_mf.db successfully!


In [4]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('bluestock_mf.db')

# Q1: Top 5 funds by AUM
print("Q1: Top 5 Funds by AUM")
print("="*50)
q1 = pd.read_sql_query("""
    SELECT scheme_name, fund_house, aum_crore
    FROM fact_performance
    ORDER BY aum_crore DESC
    LIMIT 5
""", conn)
print(q1)

# Q2: Average NAV per month for SBI Bluechip
print("\nQ2: Average NAV per month - SBI Bluechip")
print("="*50)
q2 = pd.read_sql_query("""
    SELECT strftime('%Y-%m', date) as month,
           ROUND(AVG(nav), 2) as avg_nav
    FROM fact_nav
    WHERE amfi_code = 119551
    GROUP BY month
    ORDER BY month
    LIMIT 10
""", conn)
print(q2)

# Q3: Transactions by state
print("\nQ3: Total Transactions by State")
print("="*50)
q3 = pd.read_sql_query("""
    SELECT state,
           COUNT(*) as total_transactions,
           SUM(amount_inr) as total_amount
    FROM fact_transactions
    GROUP BY state
    ORDER BY total_amount DESC
""", conn)
print(q3)

# Q4: Funds with expense ratio < 1%
print("\nQ4: Funds with Expense Ratio < 1%")
print("="*50)
q4 = pd.read_sql_query("""
    SELECT f.scheme_name, f.fund_house, p.expense_ratio_pct
    FROM dim_fund f
    JOIN fact_performance p ON f.amfi_code = p.amfi_code
    WHERE p.expense_ratio_pct < 1.0
    ORDER BY p.expense_ratio_pct ASC
""", conn)
print(q4)

# Q5: Top 5 funds by Sharpe ratio
print("\nQ5: Top 5 Funds by Sharpe Ratio")
print("="*50)
q5 = pd.read_sql_query("""
    SELECT f.scheme_name, f.fund_house, p.sharpe_ratio
    FROM dim_fund f
    JOIN fact_performance p ON f.amfi_code = p.amfi_code
    ORDER BY p.sharpe_ratio DESC
    LIMIT 5
""", conn)
print(q5)

# Q6: Transaction split by type
print("\nQ6: Transaction Split by Type")
print("="*50)
q6 = pd.read_sql_query("""
    SELECT transaction_type,
           COUNT(*) as count,
           SUM(amount_inr) as total_amount
    FROM fact_transactions
    GROUP BY transaction_type
""", conn)
print(q6)

# Q7: Average SIP amount by age group
print("\nQ7: Average SIP Amount by Age Group")
print("="*50)
q7 = pd.read_sql_query("""
    SELECT age_group,
           ROUND(AVG(amount_inr), 2) as avg_sip_amount,
           COUNT(*) as total_transactions
    FROM fact_transactions
    WHERE transaction_type = 'SIP'
    GROUP BY age_group
    ORDER BY avg_sip_amount DESC
""", conn)
print(q7)

# Q8: Top 5 funds by 3 year returns
print("\nQ8: Top 5 Funds by 3 Year Returns")
print("="*50)
q8 = pd.read_sql_query("""
    SELECT f.scheme_name, f.fund_house, p.return_3yr_pct
    FROM dim_fund f
    JOIN fact_performance p ON f.amfi_code = p.amfi_code
    ORDER BY p.return_3yr_pct DESC
    LIMIT 5
""", conn)
print(q8)

# Q9: AUM by fund house
print("\nQ9: AUM by Fund House")
print("="*50)
q9 = pd.read_sql_query("""
    SELECT fund_house,
           ROUND(aum_crore, 2) as aum_crore
    FROM fact_aum
    WHERE date = (SELECT MAX(date) FROM fact_aum)
    ORDER BY aum_crore DESC
""", conn)
print(q9)

# Q10: Top 5 funds by AUM crore
print("\nQ10: Top 5 Funds by Alpha")
print("="*50)
q10 = pd.read_sql_query("""
    SELECT f.scheme_name, f.fund_house, p.alpha
    FROM dim_fund f
    JOIN fact_performance p ON f.amfi_code = p.amfi_code
    ORDER BY p.alpha DESC
    LIMIT 5
""", conn)
print(q10)

conn.close()
print("\n✅ All 10 queries executed successfully!")

Q1: Top 5 Funds by AUM
                                         scheme_name         fund_house  \
0  Mirae Asset Emerging Bluechip Fund - Regular -...     Mirae Asset MF   
1      Kotak Emerging Equity Fund - Regular - Growth  Kotak Mahindra MF   
2     Nippon India Small Cap Fund - Regular - Growth    Nippon India MF   
3         DSP Top 100 Equity Fund - Regular - Growth    DSP Mutual Fund   
4                UTI Mid Cap Fund - Regular - Growth    UTI Mutual Fund   

   aum_crore  
0      49046  
1      47469  
2      43630  
3      41828  
4      41728  

Q2: Average NAV per month - SBI Bluechip
     month  avg_nav
0  2022-01    55.04
1  2022-02    52.43
2  2022-03    50.84
3  2022-04    51.89
4  2022-05    51.89
5  2022-06    52.68
6  2022-07    52.69
7  2022-08    53.80
8  2022-09    56.51
9  2022-10    57.88

Q3: Total Transactions by State
             state  total_transactions  total_amount
0           Punjab                2965     315780459
1       Tamil Nadu                2